# Unidad 3 – Sesión 3
## Selección de variables y creación de nuevas características

**Curso:** Minería de Datos  
**Fase CRISP-DM:** Preparación de los Datos

### Propósito
- Decidir qué variables conservar, excluir o transformar;
- Identificar variables identificadoras, irrelevantes o redundantes;
- Crear nuevas variables útiles;
- Comprender feature engineering;
- Construir un dataset analítico final.

> **Idea clave:** No todas las variables disponibles deben llegar al dataset final.

In [2]:
import pandas as pd
import numpy as np

data = {
    "cliente_id": [1, 2, 3, 4, 5],
    "edad": [25, 32, 40, 29, 36],
    "ingreso_mensual": [1500, 3200, 4200, 2100, 3800],
    "gasto_mensual": [900, 1800, 2600, 1200, 2500],
    "fecha_registro": ["2025-01-10","2025-05-15","2024-08-01","2026-01-20","2024-12-05"],
    "ciudad": ["LIMA", "CUSCO", "LIMA", "AREQUIPA", "CUSCO"],
    "telefono": ["999111222","988222333","977333444","966444555","955555666"]
}

df = pd.DataFrame(data)
df

,cliente_id,edad,ingreso_mensual,gasto_mensual,fecha_registro,ciudad,telefono
0,1,25,1500,900,2025-01-10,LIMA,999111222
1,2,32,3200,1800,2025-05-15,CUSCO,988222333
2,3,40,4200,2600,2024-08-01,LIMA,977333444
3,4,29,2100,1200,2026-01-20,AREQUIPA,966444555
4,5,36,3800,2500,2024-12-05,CUSCO,955555666


# 1. Selección de variables

Pregúntate:

> ¿Esta variable aporta información relevante para el objetivo del proyecto?

Las variables pueden:
- Conservarse;
- Excluirse;
- Transformarse;
- Utilizarse para crear nuevas variables.

# 2. Identificadores

In [4]:
df[["cliente_id", "telefono"]]

,cliente_id,telefono
0,1,999111222
1,2,988222333
2,3,977333444
3,4,966444555
4,5,955555666


`cliente_id` y `telefono` identifican registros, pero normalmente no aportan valor predictivo directo.

No tiene sentido interpretar `cliente 5 > cliente 1`.

# 3. Variables redundantes

In [5]:
df["ingreso_anual"] = df["ingreso_mensual"] * 12
df[["ingreso_mensual","ingreso_anual"]]

,ingreso_mensual,ingreso_anual
0,1500,18000
1,3200,38400
2,4200,50400
3,2100,25200
4,3800,45600


Dos variables pueden contener prácticamente la misma información. Eso puede indicar redundancia.

# 4. Correlación entre variables numéricas

In [3]:
df[["edad","ingreso_mensual","gasto_mensual","ingreso_anual"]].corr()

,edad,ingreso_mensual,gasto_mensual,ingreso_anual
edad,1.000000,0.979042,0.973879,0.979042
ingreso_mensual,0.979042,1.000000,0.988692,1.000000
gasto_mensual,0.973879,0.988692,1.000000,0.988692
ingreso_anual,0.979042,1.000000,0.988692,1.000000


Una correlación alta puede sugerir redundancia, pero no obliga automáticamente a eliminar una variable. También importa el significado de negocio.

# 5. Feature Engineering

**Feature Engineering** consiste en crear nuevas variables a partir de información existente para representar mejor el fenómeno analizado.

# 6. Variable numérica derivada

In [4]:
df["porcentaje_gasto"] = (
    df["gasto_mensual"] / df["ingreso_mensual"]
) * 100

df[["ingreso_mensual","gasto_mensual","porcentaje_gasto"]]

,ingreso_mensual,gasto_mensual,porcentaje_gasto
0,1500,900,60.000000
1,3200,1800,56.250000
2,4200,2600,61.904762
3,2100,1200,57.142857
4,3800,2500,65.789474


`porcentaje_gasto` responde: ¿qué proporción del ingreso mensual utiliza el cliente en gastos?

# 7. Transformación temporal

In [5]:
df["fecha_registro"] = pd.to_datetime(df["fecha_registro"], errors="coerce")
df["anio_registro"] = df["fecha_registro"].dt.year
df["mes_registro"] = df["fecha_registro"].dt.month

df[["fecha_registro","anio_registro","mes_registro"]]

,fecha_registro,anio_registro,mes_registro
0,2025-01-10,2025,1
1,2025-05-15,2025,5
2,2024-08-01,2024,8
3,2026-01-20,2026,1
4,2024-12-05,2024,12


# 8. Crear antigüedad

In [6]:
fecha_referencia = pd.Timestamp("2026-11-01")

df["antiguedad_dias"] = (
    fecha_referencia - df["fecha_registro"]
).dt.days

df[["fecha_registro","antiguedad_dias"]]

,fecha_registro,antiguedad_dias
0,2025-01-10,660
1,2025-05-15,535
2,2024-08-01,822
3,2026-01-20,285
4,2024-12-05,696


# 9. Discretización o Binning

In [7]:
df["grupo_edad"] = pd.cut(
    df["edad"],
    bins=[0,25,35,50,100],
    labels=["18-25","26-35","36-50","51+"],
    include_lowest=True
)

df[["edad","grupo_edad"]]

,edad,grupo_edad
0,25,18-25
1,32,26-35
2,40,36-50
3,29,26-35
4,36,36-50


La discretización transforma una variable numérica en grupos. Mejora interpretación, pero puede perder detalle.

In [8]:
df.groupby("grupo_edad", observed=False)["gasto_mensual"].agg(
    cantidad="count",
    promedio="mean",
    mediana="median"
)

,cantidad,promedio,mediana
grupo_edad,,,
18-25,1,900.0,900.0
26-35,2,1500.0,1500.0
36-50,2,2550.0,2550.0
51+,0,NaN,NaN


# 10. Variable categórica derivada

In [9]:
df["nivel_ingreso"] = pd.cut(
    df["ingreso_mensual"],
    bins=[0,2000,3500,float("inf")],
    labels=["BAJO","MEDIO","ALTO"]
)

df[["ingreso_mensual","nivel_ingreso"]]

,ingreso_mensual,nivel_ingreso
0,1500,BAJO
1,3200,MEDIO
2,4200,ALTO
3,2100,MEDIO
4,3800,ALTO


# 11. Variables agregadas

In [10]:
ventas = pd.DataFrame({
    "cliente_id": [1,1,1,2,2,3,3,3,3],
    "factura_id": ["F1","F2","F3","F4","F5","F6","F7","F8","F9"],
    "monto": [100,150,200,80,120,50,70,90,110]
})

ventas

,cliente_id,factura_id,monto
0,1,F1,100
1,1,F2,150
2,1,F3,200
3,2,F4,80
4,2,F5,120
5,3,F6,50
6,3,F7,70
7,3,F8,90
8,3,F9,110


In [11]:
resumen_cliente = ventas.groupby("cliente_id").agg(
    total_compras=("monto","sum"),
    promedio_compra=("monto","mean"),
    cantidad_compras=("factura_id","nunique")
).reset_index()

resumen_cliente

,cliente_id,total_compras,promedio_compra,cantidad_compras
0,1,450,150.0,3
1,2,200,100.0,2
2,3,320,80.0,4


Aquí cambia la granularidad:

**Antes:** una fila = una transacción  
**Después:** una fila = un cliente

# 12. Variables constantes

In [12]:
df["pais"] = "PERU"
df["pais"].value_counts()

pais
PERU    5
Name: count, dtype: int64

Una variable constante no distingue observaciones y suele aportar poca información analítica.

# 13. Variables con poca variabilidad

In [13]:
ejemplo_binario = pd.Series(["SI"] * 98 + ["NO"] * 2)
ejemplo_binario.value_counts(normalize=True) * 100

SI    98.0
NO     2.0
Name: proportion, dtype: float64

Poca variabilidad no significa eliminar automáticamente la variable. Puede representar una clase rara importante.

# 14. Matriz de selección

In [14]:
matriz_seleccion = pd.DataFrame({
    "Variable": [
        "cliente_id","edad","ingreso_mensual",
        "telefono","fecha_registro","porcentaje_gasto"
    ],
    "Rol": [
        "Identificador","Numérica","Numérica",
        "Identificador","Temporal","Derivada numérica"
    ],
    "Calidad": ["Alta"] * 6,
    "Relevancia": [
        "Baja","Alta","Alta","Baja","Alta","Alta"
    ],
    "Decision": [
        "Excluir como predictor","Conservar","Conservar",
        "Excluir","Transformar","Conservar"
    ],
    "Justificacion": [
        "Solo identifica registros",
        "Puede aportar información del cliente",
        "Representa capacidad económica",
        "No aporta valor analítico directo",
        "Es más útil transformarla a antigüedad",
        "Resume la relación ingreso/gasto"
    ]
})

matriz_seleccion

,Variable,Rol,Calidad,Relevancia,Decision,Justificacion
0,cliente_id,Identificador,Alta,Baja,Excluir como predictor,Solo identifica registros
1,edad,Numérica,Alta,Alta,Conservar,Puede aportar información del cliente
2,ingreso_mensual,Numérica,Alta,Alta,Conservar,Representa capacidad económica
3,telefono,Identificador,Alta,Baja,Excluir,No aporta valor analítico directo
4,fecha_registro,Temporal,Alta,Alta,Transformar,Es más útil transformarla a antigüedad
5,porcentaje_gasto,Derivada numérica,Alta,Alta,Conservar,Resume la relación ingreso/gasto


# 15. Dataset analítico final

In [15]:
variables_finales = [
    "edad",
    "ingreso_mensual",
    "gasto_mensual",
    "porcentaje_gasto",
    "antiguedad_dias",
    "grupo_edad",
    "nivel_ingreso",
    "ciudad"
]

df_analitico = df[variables_finales].copy()
df_analitico

,edad,ingreso_mensual,gasto_mensual,porcentaje_gasto,antiguedad_dias,grupo_edad,nivel_ingreso,ciudad
0,25,1500,900,60.000000,660,18-25,BAJO,LIMA
1,32,3200,1800,56.250000,535,26-35,MEDIO,CUSCO
2,40,4200,2600,61.904762,822,36-50,ALTO,LIMA
3,29,2100,1200,57.142857,285,26-35,MEDIO,AREQUIPA
4,36,3800,2500,65.789474,696,36-50,ALTO,CUSCO


# 16. Validación

In [16]:
df_analitico.dtypes

edad                   int64
ingreso_mensual        int64
gasto_mensual          int64
porcentaje_gasto     float64
antiguedad_dias        int64
grupo_edad          category
nivel_ingreso       category
ciudad                object
dtype: object

In [17]:
df_analitico.isnull().sum()

edad                0
ingreso_mensual     0
gasto_mensual       0
porcentaje_gasto    0
antiguedad_dias     0
grupo_edad          0
nivel_ingreso       0
ciudad              0
dtype: int64

In [18]:
df_analitico.describe(include="all")

,edad,ingreso_mensual,gasto_mensual,porcentaje_gasto,antiguedad_dias,grupo_edad,nivel_ingreso,ciudad
count,5.00000,5.000000,5.000000,5.000000,5.000000,5,5,5
unique,NaN,NaN,NaN,NaN,NaN,3,3,3
top,NaN,NaN,NaN,NaN,NaN,26-35,MEDIO,LIMA
freq,NaN,NaN,NaN,NaN,NaN,2,2,2
mean,32.40000,2960.000000,1800.000000,60.217419,599.600000,NaN,NaN,NaN
std,5.85662,1137.101579,758.287544,3.844982,203.438689,NaN,NaN,NaN
min,25.00000,1500.000000,900.000000,56.250000,285.000000,NaN,NaN,NaN
25%,29.00000,2100.000000,1200.000000,57.142857,535.000000,NaN,NaN,NaN
50%,32.00000,3200.000000,1800.000000,60.000000,660.000000,NaN,NaN,NaN
75%,36.00000,3800.000000,2500.000000,61.904762,696.000000,NaN,NaN,NaN


Verifica:
- Tipos
- Nulos
- Sentido de las nuevas variables
- Categorías
- Rangos
- Coherencia con variables originales

# 17. Actividad práctica con dataset propio

In [6]:
import pandas as pd

df_proyecto = pd.read_csv(
    "/srv/jupyter-data/Compartido/Minería/LI-Small_Trans.csv",
    dtype={"From Bank": str, "To Bank": str, "Account": str, "Account.1": str},
)
print(df_proyecto.shape)
df_proyecto.head()

(6924049, 11)


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:08,011,8000ECA90,011,8000ECA90,3195403.00,US Dollar,3195403.00,US Dollar,Reinvestment,0
1,2022/09/01 00:21,03402,80021DAD0,03402,80021DAD0,1858.96,US Dollar,1858.96,US Dollar,Reinvestment,0
2,2022/09/01 00:00,011,8000ECA90,001120,8006AA910,592571.00,US Dollar,592571.00,US Dollar,Cheque,0
3,2022/09/01 00:16,03814,8006AD080,03814,8006AD080,12.32,US Dollar,12.32,US Dollar,Reinvestment,0
4,2022/09/01 00:00,020,8006AD530,020,8006AD530,2941.56,US Dollar,2941.56,US Dollar,Reinvestment,0


In [7]:
# Transformaciones y variables nuevas sobre el dataset real
df_proyecto["Timestamp"] = pd.to_datetime(
    df_proyecto["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce"
)

# Numérica derivada: log de monto (distribución muy asimétrica)
df_proyecto["monto_pagado_log"] = np.log1p(df_proyecto["Amount Paid"])

# Temporales
df_proyecto["hora"] = df_proyecto["Timestamp"].dt.hour
df_proyecto["dia_semana"] = df_proyecto["Timestamp"].dt.day_name()

# Categórica derivada: rango de monto (la relación con el lavado no es lineal)
df_proyecto["rango_monto"] = pd.cut(
    df_proyecto["Amount Paid"],
    bins=[0, 1e3, 1e4, 1e5, float("inf")],
    labels=["BAJO", "MEDIO", "ALTO", "MUY_ALTO"],
)

# Binaria: cambio de moneda
df_proyecto["misma_moneda"] = (
    df_proyecto["Receiving Currency"] == df_proyecto["Payment Currency"]
).astype(int)

nuevas = ["monto_pagado_log", "hora", "dia_semana", "rango_monto", "misma_moneda"]
print(df_proyecto[nuevas].isnull().sum())
df_proyecto[["Timestamp"] + nuevas].head()

monto_pagado_log    0
hora                0
dia_semana          0
rango_monto         0
misma_moneda        0
dtype: int64


,Timestamp,monto_pagado_log,hora,dia_semana,rango_monto,misma_moneda
0,2022-09-01 00:08:00,14.977224,0,Thursday,MUY_ALTO,1
1,2022-09-01 00:21:00,7.528310,0,Thursday,MEDIO,1
2,2022-09-01 00:00:00,13.292228,0,Thursday,MUY_ALTO,1
3,2022-09-01 00:16:00,2.589267,0,Thursday,BAJO,1
4,2022-09-01 00:00:00,7.987035,0,Thursday,MEDIO,1


Cada grupo debe identificar:
- 2 variables que conservará;
- 2 variables que excluirá;
- 2 variables que transformará;
- mínimo 2 variables nuevas que creará.

In [8]:
matriz_proyecto = pd.DataFrame({
    "Variable": [
        "Timestamp", "From Bank / To Bank", "Account / Account.1",
        "Amount Paid", "Amount Received",
        "Receiving Currency / Payment Currency", "Payment Format", "Is Laundering",
    ],
    "Rol": [
        "Temporal", "Identificador", "Identificador",
        "Numérica", "Numérica", "Categórica", "Categórica", "Objetivo",
    ],
    "Calidad": ["Alta"] * 8,
    "Relevancia": ["Alta", "Baja", "Baja", "Alta", "Baja", "Media", "Alta", "Alta"],
    "Decision": [
        "Transformar", "Excluir", "Excluir",
        "Transformar", "Excluir", "Transformar", "Conservar", "Conservar (objetivo)",
    ],
    "Justificacion": [
        "Se extraen hora y día de semana",
        "Código de banco: identifica, no cuantifica",
        "Identifica cuenta; no aporta patrón general",
        "Se logaritmiza (monto_pagado_log) y se categoriza (rango_monto)",
        "Igual a Amount Paid en 98.6 % de filas: redundante",
        "Se resume en misma_moneda",
        "Señal fuerte: ACH concentra el lavado",
        "Variable a predecir",
    ],
})

matriz_proyecto

,Variable,Rol,Calidad,Relevancia,Decision,Justificacion
0,Timestamp,Temporal,Alta,Alta,Transformar,Se extraen hora y día de semana
1,From Bank / To Bank,Identificador,Alta,Baja,Excluir,"Código de banco: identifica, no cuantifica"
2,Account / Account.1,Identificador,Alta,Baja,Excluir,Identifica cuenta; no aporta patrón general
3,Amount Paid,Numérica,Alta,Alta,Transformar,Se logaritmiza (monto_pagado_log) y se categor...
4,Amount Received,Numérica,Alta,Baja,Excluir,Igual a Amount Paid en 98.6 % de filas: redund...
5,Receiving Currency / Payment Currency,Categórica,Alta,Media,Transformar,Se resume en misma_moneda
6,Payment Format,Categórica,Alta,Alta,Conservar,Señal fuerte: ACH concentra el lavado
7,Is Laundering,Objetivo,Alta,Alta,Conservar (objetivo),Variable a predecir


# 18. Feature Engineering obligatorio

Variables creadas sobre el dataset de transacciones:

| Tipo | Variable | Origen | Cómo se creó |
|---|---|---|---|
| Numérica derivada | `monto_pagado_log` | `Amount Paid` | `np.log1p` (reduce la asimetría de los montos) |
| Temporal | `hora`, `dia_semana` | `Timestamp` | `.dt.hour` y `.dt.day_name()` |
| Categórica derivada | `rango_monto` | `Amount Paid` | `pd.cut` en BAJO, MEDIO, ALTO y MUY_ALTO |
| Binaria | `misma_moneda` | `Receiving Currency`, `Payment Currency` | comparación `==` |

# 19. Matriz final

In [9]:
print(df_proyecto["Is Laundering"].value_counts())
print(df_proyecto.shape)

# Dataset analítico final del proyecto
variables_proyecto = ["hora", "dia_semana", "rango_monto", "misma_moneda", "Payment Format", "Is Laundering"]
df_analitico_proyecto = df_proyecto[variables_proyecto].copy()
print(df_analitico_proyecto.dtypes)
print(df_analitico_proyecto.isnull().sum())
df_analitico_proyecto.head()

Is Laundering
0    6920484
1       3565
Name: count, dtype: int64
(6924049, 16)
hora                 int32
dia_semana          object
rango_monto       category
misma_moneda         int64
Payment Format      object
Is Laundering        int64
dtype: object
hora              0
dia_semana        0
rango_monto       0
misma_moneda      0
Payment Format    0
Is Laundering     0
dtype: int64


,hora,dia_semana,rango_monto,misma_moneda,Payment Format,Is Laundering
0,0,Thursday,MUY_ALTO,1,Reinvestment,0
1,0,Thursday,MEDIO,1,Reinvestment,0
2,0,Thursday,MUY_ALTO,1,Cheque,0
3,0,Thursday,BAJO,1,Reinvestment,0
4,0,Thursday,MEDIO,1,Reinvestment,0


In [10]:
matriz_final = pd.DataFrame({
    "Variable_original": [
        "Timestamp", "Amount Paid", "Receiving Currency / Payment Currency",
        "Payment Format", "From Bank / To Bank", "Account / Account.1",
        "Amount Received", "Is Laundering",
    ],
    "Decision": [
        "Transformar", "Transformar", "Transformar",
        "Conservar", "Excluir", "Excluir", "Excluir", "Objetivo",
    ],
    "Variables_resultantes": [
        "hora, dia_semana", "monto_pagado_log, rango_monto", "misma_moneda",
        "Payment Format (One-Hot)", "—", "—", "—", "Is Laundering",
    ],
    "Motivo": [
        "Patrón horario y semanal de transacciones",
        "Log reduce asimetría; rango captura relación no lineal",
        "Cambio de moneda: sin lavado",
        "ACH concentra el lavado",
        "Identificador de banco",
        "Identificador de cuenta",
        "Redundante con Amount Paid",
        "Variable a predecir",
    ],
})

matriz_final

,Variable_original,Decision,Variables_resultantes,Motivo
0,Timestamp,Transformar,"hora, dia_semana",Patrón horario y semanal de transacciones
1,Amount Paid,Transformar,"monto_pagado_log, rango_monto",Log reduce asimetría; rango captura relación n...
2,Receiving Currency / Payment Currency,Transformar,misma_moneda,Cambio de moneda: sin lavado
3,Payment Format,Conservar,Payment Format (One-Hot),ACH concentra el lavado
4,From Bank / To Bank,Excluir,—,Identificador de banco
5,Account / Account.1,Excluir,—,Identificador de cuenta
6,Amount Received,Excluir,—,Redundante con Amount Paid
7,Is Laundering,Objetivo,Is Laundering,Variable a predecir


# 20. Producto final

Cada grupo debe entregar:
- matriz de selección;
- variables conservadas;
- variables excluidas;
- variables transformadas;
- mínimo 2 variables nuevas;
- dataset analítico final;
- conclusión de máximo 5 líneas.

> ¿Qué variables fueron seleccionadas o creadas y por qué mejoran la utilidad del dataset?
> Las transformaciones fueron necesarias porque el dataset tiene identificadores (bancos, cuentas) y montos muy asimétricos que no se pueden usar tal cual. Excluimos 3 variables redundantes o sin valor predictivo, transformamos 3 (fecha, monto y monedas) en 5 variables nuevas y conservamos el método de pago, que es la señal más fuerte del lavado (ACH). El dataset final (6,924,049 filas, 16 columnas) no tiene nulos, pero el lavado es solo el 0.05 %, así que cualquier modelo deberá manejar ese desbalance. Estas variables mejoran la utilidad del dataset porque separan el patrón temporal, el tamaño del monto y el tipo de pago que sí se asocian con transacciones sospechosas.

# 22. Cierre

### Sesión 1: Limpiar
- nulos
- duplicados
- inconsistencias
- inválidos
- outliers

### Sesión 2: Transformar
- tipos
- fechas
- categorías
- escalamiento

### Sesión 3: Seleccionar y crear
- selección de variables
- redundancia
- feature engineering
- discretización
- agregaciones
- dataset analítico final

> **Preparar datos significa construir un conjunto de variables relevante, confiable y adecuado para el objetivo del proyecto.**